In [1]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [2]:
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.decomposition import TruncatedSVD
import os
import wandb
from scipy.stats import uniform, loguniform, randint
import sys
sys.path.append(os.path.join(os.getcwd(), '..'))
from util.preprocessing import TweetPreprocessor 
import joblib
from joblib import parallel_backend

# Consts

In [3]:
MODEL_DIR = os.path.join(os.getcwd(), 'models')
DATASETS_DIR = os.path.join(os.getcwd(), 'datasets')

# Dataset

In [4]:
train_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_train.csv"))
val_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_val.csv"))
train_df = pd.concat([train_df, val_df], ignore_index=True)

In [5]:
x_train, y_train = train_df["text"], train_df["gender_label"]

# Initiate pipeline

In [6]:
pipeline = Pipeline([
    ("preprocessor", TweetPreprocessor()),
    ("features", FeatureUnion([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])

# Init wandb

In [7]:
wandbToken = os.getenv("WANDB_TOKEN")
if not wandbToken:
    raise ValueError("Please set the WANDB_TOKEN environment variable to log results to Weights & Biases.")
wandb.login(key=wandbToken)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/user/.netrc
wandb: Currently logged in as: qgurulev (who-wrote-it-nlp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Randomized search

In [8]:
ngram_ranges_word = [(1, 2), (1, 3), (2, 3)]
ngram_ranges_char = [(2, 4), (3, 5), (4, 6), (2, 5)]

rnd_params = {
    "features__tfidf_word__use_idf": [True, False],
    "features__tfidf_word__sublinear_tf": [True, False],
    "features__tfidf_word__norm": ["l1", "l2"],
    "features__tfidf_word__max_df": uniform(0.6, 0.4),
    "features__tfidf_word__min_df": uniform(0.001, 0.4),
    "features__tfidf_word__max_features": randint(5000, 120000),
    "features__tfidf_word__ngram_range": ngram_ranges_word,

    "features__tfidf_char__use_idf": [True, False],
    "features__tfidf_char__sublinear_tf": [True, False],
    "features__tfidf_char__norm": ["l1", "l2"],
    "features__tfidf_char__max_df": uniform(0.6, 0.4),
    "features__tfidf_char__min_df": uniform(0.001, 0.4),
    "features__tfidf_char__max_features": randint(5000, 120000),
    "features__tfidf_char__ngram_range": ngram_ranges_char,

    "svd__n_components": randint(10, 400),

    "clf__C": loguniform(1e-2, 1e2),
}

In [9]:
rnd_search = RandomizedSearchCV(
    pipeline, 
    rnd_params, 
    n_iter=500,
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=int(os.getenv("RANDOM_SEED", 880055535))),
    scoring="f1_macro",
    n_jobs=-1,
    verbose=10,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)


In [11]:
with parallel_backend("threading"):
    rnd_search.fit(x_train, y_train)

Fitting 4 folds for each of 500 candidates, totalling 2000 fits
[CV 2/4; 24/500] START clf__C=11.610978911168623, features__tfidf_char__max_df=0.9103379268978574, features__tfidf_char__max_features=106502, features__tfidf_char__min_df=0.35077329881909114, features__tfidf_char__ngram_range=(2, 4), features__tfidf_char__norm=l1, features__tfidf_char__sublinear_tf=False, features__tfidf_char__use_idf=False, features__tfidf_word__max_df=0.8662864041383516, features__tfidf_word__max_features=116948, features__tfidf_word__min_df=0.11848885781024542, features__tfidf_word__ngram_range=(1, 3), features__tfidf_word__norm=l1, features__tfidf_word__sublinear_tf=True, features__tfidf_word__use_idf=False, svd__n_components=242
[CV 1/4; 1/500] START clf__C=0.7358430600503051, features__tfidf_char__max_df=0.6731731574800428, features__tfidf_char__max_features=29905, features__tfidf_char__min_df=0.29409247649650583, features__tfidf_char__ngram_range=(2, 4), features__tfidf_char__norm=l1, features__tfid

KeyboardInterrupt: 

In [ ]:
rnd_run = wandb.init(project="who-wrote-it-nlp", name="hyperparam_search", entity="who-wrote-it-nlp", job_type="hyperparam_search_rnd",group="random_search")

rnd_results_df = pd.DataFrame(rnd_search.cv_results_)

for i, row in rnd_results_df.iterrows():
    params = row["params"]
    wandb.log(
        {
            **params,
            "score": row["mean_test_score"],
            "std": row["std_test_score"],
        },
    )

# Log full CV results table for interactive analysis in W&B UI
wandb.log({"rnd_cv_results": wandb.Table(dataframe=rnd_results_df)})

# Auto-log scatter plots for numeric params vs score/std
for p in rnd_params.keys():
    col = f"param_{p}"
    if col in rnd_results_df.columns and pd.api.types.is_numeric_dtype(rnd_results_df[col]):
        score_plot_df = rnd_results_df[[col, "mean_test_score"]].rename(
            columns={col: p, "mean_test_score": "score"}
        )
        score_tbl = wandb.Table(dataframe=score_plot_df)
        wandb.log({
            f"rnd_{p}_vs_score": wandb.plot.scatter(score_tbl, p, "score", title=f"[RND] {p} vs score")
        })

        std_plot_df = rnd_results_df[[col, "std_test_score"]].rename(
            columns={col: p, "std_test_score": "std"}
        )
        std_tbl = wandb.Table(dataframe=std_plot_df)
        wandb.log({
            f"rnd_{p}_vs_std": wandb.plot.scatter(std_tbl, p, "std", title=f"[RND] {p} vs std")
        })

best_rnd = rnd_search.best_params_
best_rnd_score = rnd_search.best_score_
best_rnd_model = rnd_search.best_estimator_
wandb.log({
    "best_score": best_rnd_score,
    "best_params": best_rnd
})

rnd_model_path = os.path.join(MODEL_DIR, "best_rnd_model.joblib")
joblib.dump(best_rnd_model, rnd_model_path)

artifact = wandb.Artifact("best_rnd_model", type="model")
artifact.add_file(rnd_model_path)
wandb.log_artifact(artifact)

rnd_run.finish()

# Grid search

In [ ]:
def make_range(value, pct=0.05, cast_int=False):
    factors = [1 - pct, 1, 1 + pct]

    candidates = [value * f for f in factors]

    if cast_int:
        candidates = [int(round(c)) for c in candidates]

    uniq = []
    for c in candidates:
        if c not in uniq:
            uniq.append(c)

    return uniq

grid_params = {
    "features__tfidf_word__use_idf": [best_rnd["features__tfidf_word__use_idf"]],
    "features__tfidf_word__sublinear_tf": [best_rnd["features__tfidf_word__sublinear_tf"]],
    "features__tfidf_word__norm": [best_rnd["features__tfidf_word__norm"]],
    "features__tfidf_word__ngram_range": [best_rnd["features__tfidf_word__ngram_range"]],
    "features__tfidf_word__max_df": make_range(best_rnd["features__tfidf_word__max_df"]),
    "features__tfidf_word__min_df": make_range(best_rnd["features__tfidf_word__min_df"]),
    "features__tfidf_word__max_features": make_range(best_rnd["features__tfidf_word__max_features"], cast_int=True),

    "features__tfidf_char__use_idf": [best_rnd["features__tfidf_char__use_idf"]],
    "features__tfidf_char__sublinear_tf": [best_rnd["features__tfidf_char__sublinear_tf"]],
    "features__tfidf_char__norm": [best_rnd["features__tfidf_char__norm"]],
    "features__tfidf_char__ngram_range": [best_rnd["features__tfidf_char__ngram_range"]],
    "features__tfidf_char__max_df": make_range(best_rnd["features__tfidf_char__max_df"]),
    "features__tfidf_char__min_df": make_range(best_rnd["features__tfidf_char__min_df"]),
    "features__tfidf_char__max_features": make_range(best_rnd["features__tfidf_char__max_features"], cast_int=True),

    "svd__n_components": [best_rnd["svd__n_components"]],

    "clf__C": make_range(best_rnd["clf__C"]),
}

In [ ]:
grid_search = GridSearchCV(
    pipeline, 
    grid_params,
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=int(os.getenv("RANDOM_SEED", 880055535))),
    scoring="f1_macro",
    n_jobs=-1,
    verbose=10,
)

In [ ]:
with parallel_backend("threading"):
    grid_search.fit(x_train, y_train)

In [ ]:
grid_run = wandb.init(project="who-wrote-it-nlp", name="hyperparam_search", entity="who-wrote-it-nlp", job_type="hyperparam_search_grid",group="grid_search")

results_df = pd.DataFrame(grid_search.cv_results_)

for i, row in results_df.iterrows():
    params = row["params"]
    wandb.log(
        {
            **params,
            "score": row["mean_test_score"],
            "std": row["std_test_score"],
        },
    )

# Log full CV results table for interactive analysis in W&B UI
wandb.log({"grid_cv_results": wandb.Table(dataframe=results_df)})

# Optional: auto-log scatter plots for numeric params vs score
for p in grid_params.keys():
    col = f"param_{p}"
    if col in results_df.columns and pd.api.types.is_numeric_dtype(results_df[col]):
        plot_df = results_df[[col, "mean_test_score"]].rename(
            columns={col: p, "mean_test_score": "score"}
        )
        tbl = wandb.Table(dataframe=plot_df)
        wandb.log({
            f"{p}_vs_score": wandb.plot.scatter(tbl, p, "score", title=f"{p} vs score")
        })

best_grid = grid_search.best_params_
best_grid_score = grid_search.best_score_
best_grid_model = grid_search.best_estimator_
wandb.log({
    "best_score": best_grid_score,
    "best_params": best_grid
})

grid_model_path = os.path.join(MODEL_DIR, "best_grid_model.joblib")
joblib.dump(best_grid_model, grid_model_path)
artifact = wandb.Artifact("best_grid_model", type="model")
artifact.add_file(grid_model_path)
wandb.log_artifact(artifact)

grid_run.finish()